# Anchor Selector

### Procedure

1. Load SGA-2025
2. Apply size filter
3. Apply morphology selection
4. Create sampler
5. Create correct anchor save system
6. Create alert when 1,100 galaxies of each type are classified

In [1]:
# for debugging cutout_vetter
%load_ext autoreload
%autoreload 2

In [2]:
import glob
import h5py
import os

from SGA.SGA import read_sga_sample # loading SGA-2025
from astropy.table import vstack

import pandas as pd
import numpy as np

from astropy.coordinates import SkyCoord # galaxy matching
import astropy.units as u

from SGA.qa import sdss_rgb # displaying SGA-2025 images

from cutout_vetter import CutoutVetter # interactive image grid

In [3]:
MAX_SEP = 9.5 # arcseconds
SAMPLE_SIZE = 1000
N_COLS = 10
MORPHOLOGY_CODES = {
    "Elliptical": 20,
    "Lenticular": 0,
    "Spiral": 10,
    "Irregular": -5,
}

In [4]:
def build_cutout_index(ssl_dir, verbose=False):
    """
    Return {(region, sgaid): (hdf5_path, row_index)}
    for fast image retrieval.
    """
    files = sorted(glob.glob(os.path.join(ssl_dir, "ssl-cutouts-dr11-*.hdf5")))
    if not files:
        raise FileNotFoundError(f"No cutout files found in {ssl_dir}")

    index = {}
    for f in files:
        filename = os.path.basename(f)
        
        if "dr11-south" in filename:
            region = "dr11-south"
        elif "dr11-north" in filename:
            region = "dr11-north"
        else:
            raise ValueError(f"Could not determine region from filename: {filename}")
            
        with h5py.File(f, "r") as H:
            sgaids = H["sgaid"][:]
            if verbose:
                print(f"  {filename}: {len(sgaids):,} galaxies ({region})")
            
            for i, sgaid in enumerate(sgaids):
                key = (region, int(sgaid))
                if key in index:
                    print(f"WARNING: duplicate key found: {key}")
                index[key] = (f, i)

    print(f"Total unique (region, SGAID) pairs indexed: {len(index):,}")

    return index

def generate_SGA_2025(cutout_index):
    _, south = read_sga_sample(region="dr11-south", no_groups=True)
    _, north = read_sga_sample(region="dr11-north", no_groups=True)

    south["VI_REGION"] = "dr11-south"
    north["VI_REGION"] = "dr11-north"

    catalog = vstack([south, north]).to_pandas()
    catalog["cutout_key"] = list(zip(catalog["VI_REGION"], catalog["SGAID"].astype(int)))
    catalog = catalog[catalog["cutout_key"].isin(cutout_index)].copy()
    catalog.drop(columns="cutout_key", inplace=True)
    catalog.reset_index(drop=True, inplace=True)
    
    return catalog

def selectSizeBin(df, binNum, delim=None):
    cutout_size = 152
    pixel_value = 0.262 # arcseconds
    if not delim:
        delim = cutout_size * pixel_value / 60
    df = df[(df['D26'] > binNum*delim) & (df['D26'] <= (binNum+1)*delim)]
    print(f"Selected {len(df)} galaxies less than {delim*60} arcseconds")
    return df

# Not needed in the final version
def morphSplit(df):
    grouped = df.groupby('Galaxy_Type')

    dfs = {morph: group for morph, group in grouped}
    return dfs

def SGA_2025_Predictions(preds, SGA_2025, max_sep=MAX_SEP):

    pred_coords = SkyCoord(
        ra=preds["target_ra"].to_numpy(dtype=float),
        dec=preds["target_dec"].to_numpy(dtype=float),
        unit="deg"
    )

    sample_coords = SkyCoord(
        ra=SGA_2025["RA"].to_numpy(dtype=float),
        dec=SGA_2025["DEC"].to_numpy(dtype=float),
        unit="deg"
    )

    # For every anchor, find the nearest SGA-2025 galaxy
    idx, sep2d, _ = pred_coords.match_to_catalog_sky(sample_coords)

    max_sep_u = max_sep * u.arcsec
    good = sep2d < max_sep_u

    # SGA-2025 counterparts of successfully matched predictions
    filtered_predictions = SGA_2025.iloc[idx[good]].copy()

    # Add the prediction catalog's main_type
    filtered_predictions["Galaxy_Type"] = (preds.iloc[np.where(good)[0]]["Galaxy_Type"].to_numpy())

    print(f"Matched {good.sum()} / {len(preds)} predictions")
    print("Minimum:", sep2d.min().arcsec, "arcsec")
    print("Median :", np.median(sep2d.arcsec), "arcsec")
    print("Maximum:", sep2d.max().arcsec, "arcsec")

    return filtered_predictions, idx, sep2d

def sampleMorphologies(df, n=SAMPLE_SIZE, morph_col="Galaxy_Type"):
    sample = df.groupby(morph_col).sample(n)
    return sample
    
def lookup_by_morph(morph_label, df):
    return df[df["Galaxy_Type"] == morph_label].copy()

In [5]:
SSL_DIR = '/global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl'
cutout_index = build_cutout_index(SSL_DIR)
SGA_2025 = generate_SGA_2025(cutout_index)
filtered_galaxies = selectSizeBin(SGA_2025, 1)

Total unique (region, SGAID) pairs indexed: 445,693
INFO:SGA.py:363:_read_catalog: Read 395,435/395,435 GROUP_PRIMARY objects from /dvs_ro/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/sample/SGA2025-beta-v1.6-dr11-south.fits
INFO:SGA.py:370:_read_catalog: Selecting 395,435/395,435 objects in region=dr11-south
INFO:SGA.py:363:_read_catalog: Read 90,504/90,504 GROUP_PRIMARY objects from /dvs_ro/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/sample/SGA2025-beta-v1.6-dr11-north.fits
INFO:SGA.py:370:_read_catalog: Selecting 90,504/90,504 objects in region=dr11-north
Selected 210137 galaxies less than 39.824 arcseconds


In [9]:
predictions = pd.read_csv('/pscratch/sd/q/qshimp/Sorter/binary_classifier/sga2025/classifications/03_spiral_predictions.csv')
matched_predictions, _, _ = SGA_2025_Predictions(predictions, filtered_galaxies)
#df = morphSplit(matched_predictions)

Matched 149360 / 310903 predictions
Minimum: 0.0 arcsec
Median : 119.71188408304008 arcsec
Maximum: 5372.428337226898 arcsec


In [10]:
SAMPLE_PATH = "/pscratch/sd/q/qshimp/Sorter/anchor_selector_sample.csv"

matched_predictions, _, _ = SGA_2025_Predictions(predictions, filtered_galaxies)

if os.path.exists(SAMPLE_PATH):
    sample = pd.read_csv(SAMPLE_PATH)
    print(f"Loaded existing sample: {len(sample)} galaxies from {SAMPLE_PATH}")
else:
    sample = sampleMorphologies(matched_predictions)
    os.makedirs(os.path.dirname(SAMPLE_PATH), exist_ok=True)
    sample.to_csv(SAMPLE_PATH, index=False)
    print(f"Drew new sample: {len(sample)} galaxies, saved to {SAMPLE_PATH}")

Matched 149360 / 310903 predictions
Minimum: 0.0 arcsec
Median : 119.71188408304008 arcsec
Maximum: 5372.428337226898 arcsec
Drew new sample: 5000 galaxies, saved to /pscratch/sd/q/qshimp/Sorter/anchor_selector_sample.csv


In [11]:
%matplotlib widget
vetter = CutoutVetter(
    morph_options=list(MORPHOLOGY_CODES.keys()),
    data_lookup_fn=lookup_by_morph,
    df=sample,
    cutout_index=cutout_index,
    sdss_rgb_fn=sdss_rgb,
    ncols=N_COLS,
    n_per_page=50,
    figsize_per=2,
)

In [8]:
'''
export_implicit_correct(
    df=sample,
    morph_options=list(MORPHOLOGY_CODES.keys()),
    data_lookup_fn=lookup_by_morph,
    save_dir="/pscratch/sd/q/qshimp/Sorter",
)
'''

'\nexport_implicit_correct(\n    df=sample,\n    morph_options=list(MORPHOLOGY_CODES.keys()),\n    data_lookup_fn=lookup_by_morph,\n    save_dir="/pscratch/sd/q/qshimp/Sorter",\n)\n'

### Standings
##### Elliptical
236 Elliptical
133 Lenticular
194 Spiral
27 irregular
##### Lenticular
##### Spiral
##### Irregular
81 Irregular
275 Spiral
85 Lenticular
141 Elliptical

### Total
796 Elliptical
651 Lenticular
926 Spirals
595 Irregulars